In [1]:
import os
import time
import wave
import threading
import tkinter as tk
from tkinter import ttk, filedialog, messagebox
from tkinter import font as tkfont

import pandas as pd
import whisper
import numpy as np

# For live recording
try:
    import pyaudio
    PYAUDIO_OK = True
except ImportError:
    PYAUDIO_OK = False
    print("pyaudio not installed — live recording disabled.")

In [2]:
RESULTS_CSV = "results.csv"
AUDIO_DIR   = "Audios_Processed/"

In [3]:
df = pd.read_csv(RESULTS_CSV, encoding="utf-8-sig")
print(f"Results loaded: {len(df)} rows")

Results loaded: 60 rows


In [4]:
print("Loading Whisper model")
whisper_model = whisper.load_model("medium")
print("Model ready.")

Loading Whisper model
Model ready.


In [7]:
# Backgrounds
BG_ROOT    = "#F0F2F8"      # cool light grey-blue page background
BG_CARD    = "#FFFFFF"      # pure white cards
BG_HEADER  = "#FFFFFF"      # white header
BG_INPUT   = "#E8EBF4"      # soft blue-grey input fields
BG_ROW_ALT = "#F5F6FB"      # alternating table row tint

# Accent Colors
ACCENT      = "#2979FF"     # vivid blue — primary actions, selected tabs
ACCENT_DARK = "#1565C0"     # deep blue — hover states
ACCENT_SOFT = "#E3F0FF"     # pale blue tint — backgrounds, highlights
ACCENT2     = "#00BFA5"     # cyan-green — secondary actions (Translate button)
ACCENT2_BG  = "#E0FAF6"     # light cyan tint — secondary backgrounds

# Text
TEXT_DARK   = "#1C2340"     # near-black navy — headings, primary text
TEXT_MID    = "#5A617A"     # medium blue-grey — labels, captions
TEXT_LIGHT  = "#9BA3BC"     # light grey-blue — placeholders, hints
TEXT_WHITE  = "#FFFFFF"     # white — text on colored buttons

# Status Colors
COLOR_GOOD  = "#00C853"     # vivid green — BLEU ≥ 70, success states
COLOR_MID   = "#FFB300"     # amber — BLEU 35–69, warnings
COLOR_BAD   = "#D50000"     # strong red — BLEU < 35, errors, record button
COLOR_BORDER= "#D8DCF0"     # soft blue-grey — card and input borders

FONT_H1    = ("Segoe UI", 17, "bold")
FONT_H2    = ("Segoe UI", 12, "bold")
FONT_H3    = ("Segoe UI", 10, "bold")
FONT_BODY  = ("Segoe UI", 10)
FONT_SMALL = ("Segoe UI", 9)
FONT_MONO  = ("Consolas", 10)
FONT_BIG   = ("Segoe UI", 24, "bold")

In [8]:
def bleu_color(v):
    if v >= 70: return COLOR_GOOD
    if v >= 35: return COLOR_MID
    return COLOR_BAD

def rounded_btn(parent, text, cmd, bg=ACCENT, fg=TEXT_WHITE,
                font=FONT_H3, padx=18, pady=8, width=None):
    b = tk.Button(parent, text=text, command=cmd,
                  bg=bg, fg=fg, font=font,
                  relief="flat", padx=padx, pady=pady,
                  cursor="hand2", activebackground=ACCENT_DARK,
                  activeforeground=TEXT_WHITE,
                  bd=0, highlightthickness=0)
    if width: b.config(width=width)
    return b

def card(parent, **kwargs):
    return tk.Frame(parent, bg=BG_CARD,
                    highlightbackground=COLOR_BORDER,
                    highlightthickness=1, **kwargs)

# ══════════════════════════════════════════════
#   ROOT
# ══════════════════════════════════════════════
root = tk.Tk()
root.title("Urdu–English Code-Switching Translator")
root.geometry("1120x740")
root.configure(bg=BG_ROOT)
root.resizable(True, True)

# ── Header ──────────────────────────────────
hdr = tk.Frame(root, bg=BG_CARD,
               highlightbackground=COLOR_BORDER, highlightthickness=1)
hdr.pack(fill="x")

tk.Label(hdr, text="🎙  Urdu–English Code-Switching Translator",
         font=FONT_H1, bg=BG_CARD, fg=TEXT_DARK).pack(side="left", padx=22, pady=14)

stats_lbl = tk.Label(hdr,
    text=(f"Dataset: {len(df)} files   •   "
          f"Mean BLEU: {df['bleu_score'].mean():.1f}   •   "
          f"Mean chrF: {df['chrf_score'].mean():.1f}"),
    font=FONT_SMALL, bg=BG_CARD, fg=TEXT_MID)
stats_lbl.pack(side="right", padx=22)

# ── Tabs ─────────────────────────────────────
style = ttk.Style()
style.theme_use("clam")
style.configure("TNotebook",
                background=BG_ROOT, borderwidth=0, tabmargins=0)
style.configure("TNotebook.Tab",
                background=BG_ROOT, foreground=TEXT_MID,
                padding=[18, 9], font=FONT_H3,
                borderwidth=0)
style.map("TNotebook.Tab",
          background=[("selected", BG_CARD)],
          foreground=[("selected", ACCENT)],
          expand=[("selected", [0,0,0,2])])
style.configure("Treeview",
                background=BG_CARD, fieldbackground=BG_CARD,
                foreground=TEXT_DARK, rowheight=28, font=FONT_BODY,
                borderwidth=0)
style.configure("Treeview.Heading",
                background=ACCENT_SOFT, foreground=ACCENT_DARK,
                font=FONT_H3, relief="flat")
style.map("Treeview",
          background=[("selected", ACCENT_SOFT)],
          foreground=[("selected", ACCENT_DARK)])

nb_widget = ttk.Notebook(root, style="TNotebook")
nb_widget.pack(fill="both", expand=True, padx=12, pady=10)

# ══════════════════════════════════════════════════════════
#   TAB 1 — BROWSE RESULTS
# ══════════════════════════════════════════════════════════
tab1 = tk.Frame(nb_widget, bg=BG_ROOT)
nb_widget.add(tab1, text="  📊  Browse Results  ")

# Filter bar
fbar = card(tab1, padx=14, pady=10)
fbar.pack(fill="x", padx=4, pady=(4,4))

def lbl(p, t, **kw):
    return tk.Label(p, text=t, bg=BG_CARD, fg=TEXT_MID, font=FONT_SMALL, **kw)

lbl(fbar, "Speaker:").pack(side="left", padx=(4,3))
spk_var = tk.StringVar(value="All")
ttk.Combobox(fbar, textvariable=spk_var, width=7, state="readonly", font=FONT_BODY,
             values=["All"]+sorted(df["speaker_id"].unique().tolist())).pack(side="left", padx=(0,12))

lbl(fbar, "Gender:").pack(side="left", padx=(0,3))
gen_var = tk.StringVar(value="All")
ttk.Combobox(fbar, textvariable=gen_var, width=8, state="readonly", font=FONT_BODY,
             values=["All","Male","Female"]).pack(side="left", padx=(0,12))

lbl(fbar, "Min BLEU:").pack(side="left", padx=(0,3))
bleu_var = tk.StringVar(value="0")
tk.Entry(fbar, textvariable=bleu_var, width=5, bg=BG_INPUT, fg=TEXT_DARK,
         font=FONT_BODY, relief="flat", bd=4).pack(side="left", padx=(0,12))

count_lbl = tk.Label(fbar, text="", font=FONT_SMALL, bg=BG_CARD, fg=TEXT_LIGHT)
count_lbl.pack(side="right", padx=8)

# Tree
tree_wrap = tk.Frame(tab1, bg=BG_ROOT)
tree_wrap.pack(fill="both", expand=True, padx=4)

cols = ("file","speaker","gender","bleu","chrf","lang")
hdrs = ("File","Speaker","Gender","BLEU","chrF","Detected")
widths = (90,80,80,70,70,90)

tree = ttk.Treeview(tree_wrap, columns=cols, show="headings", height=14)
for c,h,w in zip(cols,hdrs,widths):
    tree.heading(c, text=h)
    tree.column(c, width=w, anchor="center", minwidth=60)

vsb = ttk.Scrollbar(tree_wrap, orient="vertical", command=tree.yview)
tree.configure(yscrollcommand=vsb.set)
tree.pack(side="left", fill="both", expand=True)
vsb.pack(side="right", fill="y")

# Detail panel
det = card(tab1, padx=18, pady=12)
det.pack(fill="x", padx=4, pady=(4,4))

def det_row(label, row):
    tk.Label(det, text=label, font=FONT_H3, bg=BG_CARD,
             fg=TEXT_MID, width=18, anchor="w").grid(row=row, column=0, sticky="w", pady=2)
    v = tk.StringVar()
    tk.Label(det, textvariable=v, font=FONT_BODY, bg=BG_CARD,
             fg=TEXT_DARK, anchor="w", wraplength=680, justify="left").grid(
             row=row, column=1, sticky="w", padx=8)
    return v

dv_file  = det_row("File:", 0)
dv_roman = det_row("Roman Urdu:", 1)
dv_gt    = det_row("Ground Truth:", 2)
dv_out   = det_row("Whisper Output:", 3)

sc_frame = tk.Frame(det, bg=BG_CARD)
sc_frame.grid(row=4, column=0, columnspan=2, sticky="w", pady=(8,0))
d_bleu = tk.Label(sc_frame, text="BLEU: —", font=FONT_BIG, bg=BG_CARD, fg=TEXT_LIGHT)
d_bleu.pack(side="left", padx=(0,28))
d_chrf = tk.Label(sc_frame, text="chrF: —", font=FONT_BIG, bg=BG_CARD, fg=TEXT_LIGHT)
d_chrf.pack(side="left")

def on_select(e):
    sel = tree.selection()
    if not sel: return
    fname = tree.item(sel[0])["values"][0]
    r = df[df["audio_file"]==fname].iloc[0]
    dv_file.set(f"{fname}   Speaker: {r['speaker_id']}   Gender: {r['gender']}")
    dv_roman.set(r["ground_truth_roman"])
    dv_gt.set(r["ground_truth_english"])
    dv_out.set(r["whisper_english"])
    b,c = float(r["bleu_score"]), float(r["chrf_score"])
    d_bleu.config(text=f"BLEU: {b:.1f}", fg=bleu_color(b))
    d_chrf.config(text=f"chrF: {c:.1f}", fg=bleu_color(c))

tree.bind("<<TreeviewSelect>>", on_select)

def populate(data):
    tree.delete(*tree.get_children())
    for _, r in data.iterrows():
        b = float(r["bleu_score"])
        tag = "g" if b>=70 else ("m" if b>=35 else "b")
        tree.insert("","end",
            values=(r["audio_file"],r["speaker_id"],r["gender"],
                    f"{b:.1f}",f"{r['chrf_score']:.1f}",
                    r.get("detected_language","—")),
            tags=(tag,))
    tree.tag_configure("g", foreground=COLOR_GOOD)
    tree.tag_configure("m", foreground=COLOR_MID)
    tree.tag_configure("b", foreground=COLOR_BAD)
    count_lbl.config(text=f"{len(data)} files")

def apply_filter(*_):
    f = df.copy()
    if spk_var.get()!="All": f=f[f["speaker_id"]==spk_var.get()]
    if gen_var.get()!="All": f=f[f["gender"]==gen_var.get()]
    try: f=f[f["bleu_score"]>=float(bleu_var.get())]
    except: pass
    populate(f)

spk_var.trace("w",apply_filter)
gen_var.trace("w",apply_filter)
bleu_var.trace("w",apply_filter)
populate(df)

# ══════════════════════════════════════════════════════════
#   TAB 2 — UPLOAD & TRANSLATE
# ══════════════════════════════════════════════════════════
tab2 = tk.Frame(nb_widget, bg=BG_ROOT)
nb_widget.add(tab2, text="  📂  Upload & Translate  ")

ucard = card(tab2, padx=22, pady=18)
ucard.pack(fill="x", padx=12, pady=12)

tk.Label(ucard, text="Upload a WAV File",
         font=FONT_H2, bg=BG_CARD, fg=TEXT_DARK).pack(anchor="w")
tk.Label(ucard, text="Supports Urdu, English, or code-switched speech",
         font=FONT_SMALL, bg=BG_CARD, fg=TEXT_MID).pack(anchor="w", pady=(2,14))

upbf = tk.Frame(ucard, bg=BG_CARD)
upbf.pack(anchor="w")

fp_var = tk.StringVar(value="No file selected")

def pick():
    p = filedialog.askopenfilename(
        title="Select WAV file",
        filetypes=[("WAV","*.wav"),("All","*.*")])
    if p:
        fp_var.set(p)
        fp_lbl.config(fg=ACCENT)
        u_trans_btn.config(state="normal")
        clear_upload()

rounded_btn(upbf,"📂  Choose File", pick).pack(side="left")
fp_lbl = tk.Label(upbf, textvariable=fp_var, font=FONT_MONO,
                  bg=BG_CARD, fg=TEXT_LIGHT)
fp_lbl.pack(side="left", padx=12)

u_trans_btn = rounded_btn(ucard,"▶  Translate", lambda:
    threading.Thread(target=run_upload, daemon=True).start(),
    bg=ACCENT2, fg=TEXT_WHITE)
u_trans_btn.pack(anchor="w", pady=(12,0))
u_trans_btn.config(state="disabled")

u_prog = tk.Label(ucard, text="", font=FONT_BODY, bg=BG_CARD, fg=TEXT_MID)
u_prog.pack(anchor="w", pady=(6,0))

res2 = tk.Frame(tab2, bg=BG_ROOT)
res2.pack(fill="both", expand=True, padx=12, pady=(0,12))

def out_card(parent, title, col):
    c = card(parent, padx=14, pady=12)
    c.grid(row=0, column=col, sticky="nsew", padx=5)
    parent.columnconfigure(col, weight=1)
    parent.rowconfigure(0, weight=1)
    tk.Label(c, text=title, font=FONT_H3, bg=BG_CARD, fg=TEXT_MID).pack(anchor="w")
    t = tk.Text(c, font=FONT_BODY, bg=BG_INPUT, fg=TEXT_DARK,
                relief="flat", wrap="word", height=7,
                insertbackground=TEXT_DARK, state="disabled",
                padx=8, pady=6, bd=0)
    t.pack(fill="both", expand=True, pady=(6,0))
    return t

u_out1 = out_card(res2,"📝  Urdu Transcription",0)
u_out2 = out_card(res2,"🌐  English Translation",1)

sc2 = card(tab2, padx=18, pady=10)
sc2.pack(fill="x", padx=12, pady=(0,8))
u_score = tk.Label(sc2, text="", font=FONT_BIG, bg=BG_CARD, fg=TEXT_LIGHT)
u_score.pack(side="left", padx=(0,24))
u_lang  = tk.Label(sc2, text="", font=FONT_H3,  bg=BG_CARD, fg=TEXT_MID)
u_lang.pack(side="left")
u_time  = tk.Label(sc2, text="", font=FONT_SMALL,bg=BG_CARD, fg=TEXT_LIGHT)
u_time.pack(side="right")

def set_txt(w,txt):
    w.config(state="normal"); w.delete(1.0,"end")
    w.insert("end",txt); w.config(state="disabled")

def clear_upload():
    for w in [u_out1,u_out2]: set_txt(w,"")
    u_score.config(text="",fg=TEXT_LIGHT)
    u_lang.config(text=""); u_time.config(text=""); u_prog.config(text="")

def run_upload():
    path = fp_var.get()
    if not os.path.exists(path):
        messagebox.showerror("Error","File not found."); return
    u_trans_btn.config(state="disabled")
    u_prog.config(text="⏳  Transcribing...")
    root.update_idletasks()
    try:
        t0 = time.time()
        r_ur = whisper_model.transcribe(path, language="ur")
        set_txt(u_out1, r_ur["text"].strip())
        u_prog.config(text="⏳  Translating to English...")
        root.update_idletasks()
        r_en = whisper_model.transcribe(path, task="translate")
        set_txt(u_out2, r_en["text"].strip())
        elapsed = time.time()-t0
        fname = os.path.basename(path)
        match = df[df["audio_file"]==fname]
        if not match.empty:
            b,c = float(match.iloc[0]["bleu_score"]),float(match.iloc[0]["chrf_score"])
            u_score.config(text=f"BLEU: {b:.1f}   chrF: {c:.1f}",fg=bleu_color(b))
        else:
            u_score.config(text="New file — no reference score",fg=TEXT_MID)
        u_lang.config(text=f"Detected: {r_ur.get('language','?').upper()}", fg=ACCENT)
        u_time.config(text=f"⏱ {elapsed:.1f}s")
        u_prog.config(text="✅  Done")
    except Exception as e:
        u_prog.config(text=f"❌  {e}")
        messagebox.showerror("Error",str(e))
    finally:
        u_trans_btn.config(state="normal")

# ══════════════════════════════════════════════════════════
#   TAB 3 — LIVE RECORDING
# ══════════════════════════════════════════════════════════
tab3 = tk.Frame(nb_widget, bg=BG_ROOT)
nb_widget.add(tab3, text="  🎙  Live Record  ")

rec_card = card(tab3, padx=22, pady=18)
rec_card.pack(fill="x", padx=12, pady=12)

tk.Label(rec_card, text="Live Microphone Recording",
         font=FONT_H2, bg=BG_CARD, fg=TEXT_DARK).pack(anchor="w")
tk.Label(rec_card, text="Speak in Urdu, English, or mixed — then click Stop to translate",
         font=FONT_SMALL, bg=BG_CARD, fg=TEXT_MID).pack(anchor="w", pady=(2,14))

if not PYAUDIO_OK:
    tk.Label(rec_card,
             text="⚠️  pyaudio not installed. Run:  pip install pyaudio",
             font=FONT_H3, bg=BG_CARD, fg=COLOR_BAD).pack(anchor="w")

rec_btn_frame = tk.Frame(rec_card, bg=BG_CARD)
rec_btn_frame.pack(anchor="w")

is_recording   = threading.Event()
recorded_frames= []
REC_FILE       = "live_recording.wav"
CHUNK, RATE, CH= 1024, 16000, 1

rec_indicator = tk.Label(rec_card, text="", font=FONT_H3, bg=BG_CARD, fg=COLOR_BAD)
rec_indicator.pack(anchor="w", pady=(8,0))

rec_timer_var = tk.StringVar(value="")
tk.Label(rec_card, textvariable=rec_timer_var, font=FONT_BIG,
         bg=BG_CARD, fg=ACCENT).pack(anchor="w")

def blink_recording():
    if is_recording.is_set():
        cur = rec_indicator.cget("text")
        rec_indicator.config(
            text="● REC" if cur=="" or cur=="○ REC" else "○ REC",
            fg=COLOR_BAD if "●" in (cur or "●") else "#FFAAAA")
        root.after(600, blink_recording)

def update_timer(start):
    if is_recording.is_set():
        elapsed = time.time()-start
        rec_timer_var.set(f"{int(elapsed//60):02d}:{int(elapsed%60):02d}")
        root.after(1000, lambda: update_timer(start))

def do_record():
    global recorded_frames
    recorded_frames=[]
    p = pyaudio.PyAudio()
    stream = p.open(format=pyaudio.paInt16, channels=CH,
                    rate=RATE, input=True, frames_per_buffer=CHUNK)
    while is_recording.is_set():
        recorded_frames.append(stream.read(CHUNK, exception_on_overflow=False))
    stream.stop_stream(); stream.close(); p.terminate()
    wf = wave.open(REC_FILE,"wb")
    wf.setnchannels(CH); wf.setsampwidth(2); wf.setframerate(RATE)
    wf.writeframes(b"".join(recorded_frames)); wf.close()

def start_rec():
    if not PYAUDIO_OK:
        messagebox.showerror("Error","Install pyaudio first: pip install pyaudio")
        return
    is_recording.set()
    recorded_frames.clear()
    rec_timer_var.set("00:00")
    rec_indicator.config(text="● REC")
    start_btn.config(state="disabled")
    stop_btn.config(state="normal")
    r_out1.config(state="normal"); r_out1.delete(1.0,"end"); r_out1.config(state="disabled")
    r_out2.config(state="normal"); r_out2.delete(1.0,"end"); r_out2.config(state="disabled")
    r_score.config(text="",fg=TEXT_LIGHT); r_prog.config(text="")
    t_start = time.time()
    threading.Thread(target=do_record, daemon=True).start()
    blink_recording()
    update_timer(t_start)

def stop_rec():
    is_recording.clear()
    start_btn.config(state="normal")
    stop_btn.config(state="disabled")
    rec_indicator.config(text="⏹  Stopped", fg=TEXT_MID)
    rec_timer_var.set("")
    r_prog.config(text="⏳  Processing recording...")
    root.update_idletasks()
    threading.Thread(target=run_rec_translate, daemon=True).start()

start_btn = rounded_btn(rec_btn_frame,"⏺  Start Recording", start_rec,
                         bg=COLOR_BAD, fg=TEXT_WHITE)
start_btn.pack(side="left", padx=(0,10))

stop_btn = rounded_btn(rec_btn_frame,"⏹  Stop & Translate", stop_rec,
                        bg=TEXT_MID, fg=TEXT_WHITE)
stop_btn.config(state="disabled", disabledforeground=TEXT_WHITE)
stop_btn.pack(side="left")
stop_btn.config(state="disabled")

r_prog = tk.Label(rec_card, text="", font=FONT_BODY, bg=BG_CARD, fg=TEXT_MID)
r_prog.pack(anchor="w", pady=(8,0))

res3 = tk.Frame(tab3, bg=BG_ROOT)
res3.pack(fill="both", expand=True, padx=12, pady=(0,12))

r_out1 = out_card(res3,"📝  Urdu Transcription",0)
r_out2 = out_card(res3,"🌐  English Translation",1)

sc3 = card(tab3, padx=18, pady=10)
sc3.pack(fill="x", padx=12, pady=(0,8))
r_score = tk.Label(sc3, text="", font=FONT_BIG, bg=BG_CARD, fg=TEXT_LIGHT)
r_score.pack(side="left", padx=(0,24))
r_lang  = tk.Label(sc3, text="", font=FONT_H3, bg=BG_CARD, fg=TEXT_MID)
r_lang.pack(side="left")
r_time  = tk.Label(sc3, text="", font=FONT_SMALL, bg=BG_CARD, fg=TEXT_LIGHT)
r_time.pack(side="right")

def run_rec_translate():
    if not os.path.exists(REC_FILE):
        r_prog.config(text="❌  Recording file not found."); return
    try:
        t0 = time.time()
        r_ur = whisper_model.transcribe(REC_FILE, language="ur")
        set_txt(r_out1, r_ur["text"].strip())
        r_prog.config(text="⏳  Translating...")
        root.update_idletasks()
        r_en = whisper_model.transcribe(REC_FILE, task="translate")
        set_txt(r_out2, r_en["text"].strip())
        elapsed = time.time()-t0
        r_score.config(text="Live recording — no reference BLEU", fg=TEXT_MID)
        r_lang.config(text=f"Detected: {r_ur.get('language','?').upper()}", fg=ACCENT)
        r_time.config(text=f"⏱ {elapsed:.1f}s")
        r_prog.config(text="✅  Done")
    except Exception as e:
        r_prog.config(text=f"❌  {e}")

# ══════════════════════════════════════════════════════════
#   TAB 4 — STATISTICS
# ══════════════════════════════════════════════════════════
tab4 = tk.Frame(nb_widget, bg=BG_ROOT)
nb_widget.add(tab4, text="  📈  Statistics  ")

# Summary cards
sc_row = tk.Frame(tab4, bg=BG_ROOT)
sc_row.pack(fill="x", padx=12, pady=(14,6))

cards_data = [
    ("Total Files",    str(len(df)),                        ACCENT),
    ("Mean BLEU",      f"{df['bleu_score'].mean():.1f}",    COLOR_GOOD),
    ("Mean chrF",      f"{df['chrf_score'].mean():.1f}",    ACCENT2),
    ("BLEU = 100",     str((df['bleu_score']==100).sum()),  COLOR_GOOD),
    ("BLEU < 20",      str((df['bleu_score']< 20).sum()),   COLOR_BAD),
]
for title,val,color in cards_data:
    c = card(sc_row, padx=16, pady=14)
    c.pack(side="left", padx=5, expand=True, fill="x")
    tk.Label(c, text=val, font=("Segoe UI",22,"bold"),
             bg=BG_CARD, fg=color).pack()
    tk.Label(c, text=title, font=FONT_SMALL,
             bg=BG_CARD, fg=TEXT_MID).pack()

# Per-speaker table
spk_card = card(tab4, padx=18, pady=14)
spk_card.pack(fill="x", padx=12, pady=6)

tk.Label(spk_card, text="Per-Speaker Performance",
         font=FONT_H2, bg=BG_CARD, fg=TEXT_DARK).grid(
         row=0, column=0, columnspan=6, sticky="w", pady=(0,10))

for ci,h in enumerate(["Speaker","Gender","Files","Mean BLEU","Mean chrF","Avg Time(s)"]):
    tk.Label(spk_card, text=h, font=FONT_H3, bg=BG_CARD,
             fg=TEXT_MID, width=14).grid(row=1,column=ci,padx=4,pady=2)

spk_stats = df.groupby(["speaker_id","gender"]).agg(
    files=("audio_file","count"),
    mean_bleu=("bleu_score","mean"),
    mean_chrf=("chrf_score","mean"),
    mean_time=("processing_time_s","mean")
).reset_index().round(2)

for ri,r in spk_stats.iterrows():
    vals=[r["speaker_id"],r["gender"],int(r["files"]),
          f"{r['mean_bleu']:.1f}",f"{r['mean_chrf']:.1f}",f"{r['mean_time']:.1f}"]
    for ci,v in enumerate(vals):
        fg = bleu_color(r["mean_bleu"]) if ci==3 else TEXT_DARK
        tk.Label(spk_card,text=str(v),font=FONT_BODY,
                 bg=BG_CARD,fg=fg,width=14).grid(row=ri+2,column=ci,padx=4,pady=3)

# Gender summary
gen_card = card(tab4, padx=18, pady=14)
gen_card.pack(fill="x", padx=12, pady=6)

tk.Label(gen_card, text="Gender Comparison",
         font=FONT_H2, bg=BG_CARD, fg=TEXT_DARK).grid(
         row=0,column=0,columnspan=4,sticky="w",pady=(0,10))

for ci,h in enumerate(["Gender","Files","Mean BLEU","Mean chrF"]):
    tk.Label(gen_card,text=h,font=FONT_H3,bg=BG_CARD,
             fg=TEXT_MID,width=16).grid(row=1,column=ci,padx=4,pady=2)

gen_stats = df.groupby("gender").agg(
    files=("audio_file","count"),
    mean_bleu=("bleu_score","mean"),
    mean_chrf=("chrf_score","mean")
).reset_index().round(2)

for ri,r in gen_stats.iterrows():
    vals=[r["gender"],int(r["files"]),f"{r['mean_bleu']:.1f}",f"{r['mean_chrf']:.1f}"]
    for ci,v in enumerate(vals):
        fg = bleu_color(r["mean_bleu"]) if ci==2 else TEXT_DARK
        tk.Label(gen_card,text=str(v),font=FONT_BODY,
                 bg=BG_CARD,fg=fg,width=16).grid(row=ri+2,column=ci,padx=4,pady=3)

# ══════════════════════════════════════
root.mainloop()

C:\Users\Microsoft\AppData\Roaming\Python\Python312\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
